# SRGAN + Transfer Learning on OCT Images
**ECGR-4116/5116: AI for Biomedical Applications**

This notebook:
1. Trains **Model A** — ResNet50 transfer learning baseline on 128×128 OCT images
2. Trains an **SRGAN** to super-resolve 32×32 → 128×128 images
3. Trains **Model B** — ResNet50 on SRGAN-generated images (150+ epochs)
4. Compares both models using F1, Accuracy, and AUC

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, accuracy_score, roc_auc_score, roc_curve
)
from sklearn.utils import shuffle
import cv2
from glob import glob
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---- CHANGE THIS to your local data folder ----
DATA_DIR = './data'  # expects Data/train/AMD, Data/train/DME, Data/test/AMD, Data/test/DME

IMG_SIZE   = 128   # full resolution
LR_SIZE    = 32    # low-resolution input for SRGAN
BATCH_SIZE = 32    # small enough for local GPU/CPU
CLASSES    = ['AMD', 'DME']
print('Setup complete.')

## 2. Data Loading, Split & Visualization

In [ ]:
def load_images(data_dir, img_size=128):
    """Load images from train/ subfolders and return arrays + labels."""
    images, labels = [], []
    for label_idx, cls in enumerate(CLASSES):
        folder = os.path.join(data_dir, 'train', cls)
        paths  = glob(os.path.join(folder, '*.*'))
        print(f'  {cls}: {len(paths)} images found')
        for p in paths:
            img = cv2.imread(p)
            if img is None:
                continue
            # Convert grayscale → 3-channel if needed
            if len(img.shape) == 2:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            images.append(img)
            labels.append(label_idx)
    images = np.array(images, dtype=np.float32) / 255.0  # normalize to [0,1]
    labels = np.array(labels)
    return images, labels

def load_test_images(data_dir, img_size=128):
    """Load test set images."""
    images, labels = [], []
    for label_idx, cls in enumerate(CLASSES):
        folder = os.path.join(data_dir, 'test', cls)
        paths  = glob(os.path.join(folder, '*.*'))
        print(f'  {cls}: {len(paths)} test images')
        for p in paths:
            img = cv2.imread(p)
            if img is None:
                continue
            if len(img.shape) == 2:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            images.append(img)
            labels.append(label_idx)
    return np.array(images, dtype=np.float32) / 255.0, np.array(labels)

print('Loading training images...')
X_all, y_all = load_images(DATA_DIR, IMG_SIZE)
print(f'Total loaded: {len(X_all)} images, shape: {X_all.shape}')

# 70/30 train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
# Further split training → 80% train / 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=SEED, stratify=y_train
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

In [ ]:
# Visualize sample images with augmentation preview
aug_gen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Top row: original samples | Bottom row: augmented samples', fontsize=13)

for col in range(6):
    idx = np.random.randint(len(X_train))
    orig = X_train[idx]
    aug  = aug_gen.random_transform(orig)
    cls_name = CLASSES[y_train[idx]]

    axes[0, col].imshow(orig)
    axes[0, col].set_title(cls_name, fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(np.clip(aug, 0, 1))
    axes[1, col].set_title('augmented', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('augmentation_samples.png', dpi=100)
plt.show()
print('Augmentation samples saved.')

## 3. Model A — ResNet50 Transfer Learning (128×128)

In [ ]:
def build_resnet_classifier(input_shape=(128, 128, 3), num_classes=1, trainable_base=False):
    """
    ResNet50 with ImageNet weights.
    Top layers replaced for binary classification.
    trainable_base=False freezes the convolutional base (feature extraction only).
    """
    base = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = trainable_base  # freeze base for transfer learning

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)           # dropout to prevent overfitting
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # binary output

    model = keras.Model(inputs, outputs)
    return model

model_A = build_resnet_classifier(input_shape=(IMG_SIZE, IMG_SIZE, 3))
model_A.summary()

In [ ]:
# Compile Model A
# Adam optimizer: adaptive LR, well-suited for transfer learning
# LR=1e-4: conservative to avoid overwriting pretrained weights
model_A.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

os.makedirs('checkpoints', exist_ok=True)

callbacks_A = [
    # Stop early if val_loss doesn't improve for 10 epochs
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    # Save best model based on val_accuracy
    ModelCheckpoint('checkpoints/model_A_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    # Reduce LR when val_loss plateaus
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-7)
]

# Data augmentation generator for training
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_gen_A = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED)

# Train Model A — max 50 epochs, early stopping will kick in
print('Training Model A...')
history_A = model_A.fit(
    train_gen_A,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=callbacks_A,
    verbose=1
)
print('Model A training complete.')

In [ ]:
# Plot Model A training curves
def plot_history(history, title='Model'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{title} Training Curves', fontsize=13)

    axes[0].plot(history.history['loss'],     label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['accuracy'],     label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}_curves.png', dpi=100)
    plt.show()

plot_history(history_A, 'Model A')

## 4. SRGAN — Super-Resolution (32×32 → 128×128)

In [ ]:
# ---- SRGAN Architecture ----
# Generator: upscales LR (32x32) → HR (128x128) using residual blocks + PixelShuffle
# Discriminator: classifies real HR vs fake SR images

def residual_block(x, filters=64):
    """Residual block used in SRGAN generator."""
    skip = x
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.PReLU(shared_axes=[1, 2])(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    return layers.Add()([skip, x])

def upsample_block(x, filters=256):
    """Upsampling block: Conv → PixelShuffle (x2)."""
    x = layers.Conv2D(filters, 3, padding='same')(x)
    # DepthToSpace = PixelShuffle
    x = layers.Lambda(lambda t: tf.nn.depth_to_space(t, 2))(x)
    x = layers.PReLU(shared_axes=[1, 2])(x)
    return x

def build_generator(lr_shape=(32, 32, 3), n_residual=8):
    """
    SRGAN Generator.
    Input: 32x32 low-resolution image
    Output: 128x128 super-resolved image (4x upscale via 2 upsample blocks)
    """
    inp = keras.Input(shape=lr_shape)
    x   = layers.Conv2D(64, 9, padding='same')(inp)
    x   = layers.PReLU(shared_axes=[1, 2])(x)
    skip = x

    for _ in range(n_residual):
        x = residual_block(x, 64)

    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.Add()([skip, x])

    # 2× upsample blocks → 32→64→128
    x = upsample_block(x, 256)
    x = upsample_block(x, 256)

    outputs = layers.Conv2D(3, 9, padding='same', activation='tanh')(x)
    return keras.Model(inp, outputs, name='Generator')

def build_discriminator(hr_shape=(128, 128, 3)):
    """
    SRGAN Discriminator.
    Classifies 128x128 images as real or generated.
    Uses strided convolutions to downsample.
    """
    def disc_block(x, filters, stride=1, bn=True):
        x = layers.Conv2D(filters, 3, stride, padding='same')(x)
        if bn:
            x = layers.BatchNormalization(momentum=0.8)(x)
        x = layers.LeakyReLU(0.2)(x)
        return x

    inp = keras.Input(shape=hr_shape)
    x   = disc_block(inp, 64,  1, bn=False)
    x   = disc_block(x,   64,  2)
    x   = disc_block(x,  128,  1)
    x   = disc_block(x,  128,  2)
    x   = disc_block(x,  256,  1)
    x   = disc_block(x,  256,  2)
    x   = disc_block(x,  512,  1)
    x   = disc_block(x,  512,  2)
    x   = layers.Flatten()(x)
    x   = layers.Dense(1024)(x)
    x   = layers.LeakyReLU(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out, name='Discriminator')

generator     = build_generator(lr_shape=(LR_SIZE, LR_SIZE, 3))
discriminator = build_discriminator(hr_shape=(IMG_SIZE, IMG_SIZE, 3))

generator.summary()
discriminator.summary()

In [ ]:
# VGG19 feature extractor for perceptual loss
from tensorflow.keras.applications import VGG19

vgg = VGG19(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
vgg.trainable = False
# Use features from block3_conv3 (intermediate layer for perceptual similarity)
feature_extractor = keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer('block3_conv3').output,
    name='VGG_FeatureExtractor'
)

# Prepare LR/HR pairs from training data
# HR = 128x128 (X_train already), LR = downscaled to 32x32 then back to float
print('Preparing LR/HR pairs for SRGAN...')
X_hr = X_train.copy()  # shape (N, 128, 128, 3), values [0,1]
X_lr = np.array([
    cv2.resize(img, (LR_SIZE, LR_SIZE)) for img in X_hr
], dtype=np.float32)   # shape (N, 32, 32, 3)

# SRGAN expects inputs in [-1, 1] for tanh output
X_hr_scaled = X_hr * 2.0 - 1.0
X_lr_scaled = X_lr * 2.0 - 1.0

print(f'LR shape: {X_lr.shape}, HR shape: {X_hr.shape}')

In [ ]:
# SRGAN training loop
# Using separate optimizers for generator and discriminator
gen_optimizer  = keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9)
disc_optimizer = keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9)

bce_loss = keras.losses.BinaryCrossentropy(from_logits=False)
mse_loss = keras.losses.MeanSquaredError()

@tf.function
def train_step_srgan(lr_imgs, hr_imgs):
    batch_size = tf.shape(lr_imgs)[0]
    real_labels = tf.ones( (batch_size, 1))
    fake_labels = tf.zeros((batch_size, 1))

    # --- Train Discriminator ---
    with tf.GradientTape() as disc_tape:
        fake_hr      = generator(lr_imgs, training=True)
        real_pred    = discriminator(hr_imgs,  training=True)
        fake_pred    = discriminator(fake_hr,  training=True)
        d_loss_real  = bce_loss(real_labels, real_pred)
        d_loss_fake  = bce_loss(fake_labels, fake_pred)
        d_loss       = d_loss_real + d_loss_fake
    disc_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    # --- Train Generator ---
    with tf.GradientTape() as gen_tape:
        fake_hr        = generator(lr_imgs, training=True)
        fake_pred      = discriminator(fake_hr, training=False)
        # Adversarial loss: fool the discriminator
        adv_loss       = bce_loss(real_labels, fake_pred)
        # Perceptual (VGG) loss: match feature maps of real and fake
        # Scale from [-1,1] to [0,255] for VGG
        hr_vgg         = (hr_imgs + 1.0) * 127.5
        fake_vgg       = (fake_hr + 1.0) * 127.5
        real_features  = feature_extractor(hr_vgg,   training=False)
        fake_features  = feature_extractor(fake_vgg, training=False)
        percep_loss    = mse_loss(real_features, fake_features)
        # Total generator loss: weighted sum
        g_loss         = 1e-3 * adv_loss + percep_loss
    gen_grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))

    return d_loss, g_loss

# Training configuration
SRGAN_EPOCHS      = 50    # increase to 100+ for better results if time allows
SAVE_EVERY_N      = 5     # save checkpoint every N epochs
os.makedirs('checkpoints/srgan', exist_ok=True)

d_losses, g_losses = [], []
dataset = tf.data.Dataset.from_tensor_slices((X_lr_scaled, X_hr_scaled))\
            .shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f'Starting SRGAN training for {SRGAN_EPOCHS} epochs...')
for epoch in range(1, SRGAN_EPOCHS + 1):
    epoch_d, epoch_g = [], []
    for lr_batch, hr_batch in dataset:
        d_l, g_l = train_step_srgan(lr_batch, hr_batch)
        epoch_d.append(float(d_l))
        epoch_g.append(float(g_l))

    mean_d = np.mean(epoch_d)
    mean_g = np.mean(epoch_g)
    d_losses.append(mean_d)
    g_losses.append(mean_g)

    if epoch % SAVE_EVERY_N == 0 or epoch == 1:
        generator.save(f'checkpoints/srgan/generator_epoch_{epoch:03d}.keras')
        discriminator.save(f'checkpoints/srgan/discriminator_epoch_{epoch:03d}.keras')
        print(f'Epoch {epoch:3d}/{SRGAN_EPOCHS} | D loss: {mean_d:.4f} | G loss: {mean_g:.4f} [saved]')
    else:
        print(f'Epoch {epoch:3d}/{SRGAN_EPOCHS} | D loss: {mean_d:.4f} | G loss: {mean_g:.4f}')

print('SRGAN training complete.')

In [ ]:
# Plot SRGAN losses
plt.figure(figsize=(10, 4))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.title('SRGAN Training Losses')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('srgan_losses.png', dpi=100)
plt.show()

In [ ]:
# Show examples: LR input vs SRGAN output vs original HR
n_examples = 6
sample_idx = np.random.choice(len(X_lr), n_examples, replace=False)

lr_samples = X_lr_scaled[sample_idx]
sr_samples = generator.predict(lr_samples, verbose=0)
hr_samples = X_hr_scaled[sample_idx]

# Rescale from [-1,1] back to [0,1] for display
def rescale(imgs):
    return np.clip((imgs + 1.0) / 2.0, 0, 1)

fig, axes = plt.subplots(3, n_examples, figsize=(16, 7))
fig.suptitle('Row 1: LR (32×32) | Row 2: SRGAN Output (128×128) | Row 3: Original HR (128×128)', fontsize=11)

for col in range(n_examples):
    # LR — upscale for display
    lr_disp = cv2.resize(rescale(lr_samples[col]), (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_NEAREST)
    axes[0, col].imshow(lr_disp);              axes[0, col].axis('off')
    axes[1, col].imshow(rescale(sr_samples[col])); axes[1, col].axis('off')
    axes[2, col].imshow(rescale(hr_samples[col])); axes[2, col].axis('off')

axes[0, 0].set_ylabel('LR input',    fontsize=10)
axes[1, 0].set_ylabel('SRGAN SR',    fontsize=10)
axes[2, 0].set_ylabel('Original HR', fontsize=10)
plt.tight_layout()
plt.savefig('srgan_examples.png', dpi=100)
plt.show()

## 5. Model B — ResNet50 trained on SRGAN-generated images (150+ epochs)

In [ ]:
# Generate SR versions of ALL training images for Model B
print('Generating SR images for Model B training set...')
SR_BATCH = 64
X_sr_train = []
for i in range(0, len(X_lr_scaled), SR_BATCH):
    batch = X_lr_scaled[i:i+SR_BATCH]
    sr    = generator.predict(batch, verbose=0)
    X_sr_train.append(sr)
X_sr_train = np.concatenate(X_sr_train, axis=0)
# Rescale to [0,1]
X_sr_train = np.clip((X_sr_train + 1.0) / 2.0, 0, 1).astype(np.float32)

# Generate SR validation set too
X_val_lr = np.array([cv2.resize(img, (LR_SIZE, LR_SIZE)) for img in X_val], dtype=np.float32)
X_val_lr_scaled = X_val_lr * 2.0 - 1.0
X_sr_val = generator.predict(X_val_lr_scaled, batch_size=SR_BATCH, verbose=0)
X_sr_val = np.clip((X_sr_val + 1.0) / 2.0, 0, 1).astype(np.float32)

print(f'SR train: {X_sr_train.shape}, SR val: {X_sr_val.shape}')

In [ ]:
# Build and compile Model B (same architecture as Model A)
model_B = build_resnet_classifier(input_shape=(IMG_SIZE, IMG_SIZE, 3))

model_B.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_B = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ModelCheckpoint('checkpoints/model_B_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    # Also save every 10 epochs (handles Colab 12hr limit)
    ModelCheckpoint('checkpoints/model_B_epoch_{epoch:03d}.keras',
                    save_freq='epoch', period=10, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1, min_lr=1e-7)
]

# Augmentation for SR images
train_gen_B = train_datagen.flow(X_sr_train, y_train, batch_size=BATCH_SIZE, seed=SEED)

print('Training Model B (150 epochs)...')
history_B = model_B.fit(
    train_gen_B,
    steps_per_epoch=len(X_sr_train) // BATCH_SIZE,
    epochs=150,           # minimum 150 as required
    validation_data=(X_sr_val, y_val),
    callbacks=callbacks_B,
    verbose=1
)
print('Model B training complete.')

In [ ]:
plot_history(history_B, 'Model B')

## 6. Evaluation — Confusion Matrix, F1, Accuracy, AUC

In [ ]:
def evaluate_model(model, X_test, y_test, model_name='Model', threshold=0.5):
    """Compute and display all evaluation metrics."""
    y_prob = model.predict(X_test, verbose=0).ravel()   # probabilities
    y_pred = (y_prob >= threshold).astype(int)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    cm   = confusion_matrix(y_test, y_pred)

    print(f'\n===== {model_name} =====')
    print(f'Accuracy : {acc:.4f}')
    print(f'F1 Score : {f1:.4f}')
    print(f'AUC-ROC  : {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=CLASSES))

    # Confusion matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_confusion_matrix.png', dpi=100)
    plt.show()

    return acc, f1, auc, y_prob

# Evaluate Model A on original test images
acc_A, f1_A, auc_A, prob_A = evaluate_model(model_A, X_test, y_test, 'Model A')

# Generate SR test images for Model B
print('Generating SR test images for Model B...')
X_test_lr = np.array([cv2.resize(img, (LR_SIZE, LR_SIZE)) for img in X_test], dtype=np.float32)
X_test_lr_scaled = X_test_lr * 2.0 - 1.0
X_sr_test = generator.predict(X_test_lr_scaled, batch_size=SR_BATCH, verbose=0)
X_sr_test = np.clip((X_sr_test + 1.0) / 2.0, 0, 1).astype(np.float32)

# Evaluate Model B on SR test images
acc_B, f1_B, auc_B, prob_B = evaluate_model(model_B, X_sr_test, y_test, 'Model B')

In [ ]:
# Side-by-side ROC curves
fpr_A, tpr_A, _ = roc_curve(y_test, prob_A)
fpr_B, tpr_B, _ = roc_curve(y_test, prob_B)

plt.figure(figsize=(7, 5))
plt.plot(fpr_A, tpr_A, label=f'Model A (AUC = {auc_A:.3f})', linewidth=2)
plt.plot(fpr_B, tpr_B, label=f'Model B (AUC = {auc_B:.3f})', linewidth=2)
plt.plot([0,1],[0,1], 'k--', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model A vs Model B')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=100)
plt.show()

# Summary comparison table
print('\n========== Final Comparison ==========')
print(f'{"Metric":<12} {"Model A":>10} {"Model B":>10}')
print('-' * 34)
print(f'{"Accuracy":<12} {acc_A:>10.4f} {acc_B:>10.4f}')
print(f'{"F1 Score":<12} {f1_A:>10.4f} {f1_B:>10.4f}')
print(f'{"AUC-ROC":<12} {auc_A:>10.4f} {auc_B:>10.4f}')
print('======================================')